# Notebook 02: Preprocesamiento de Datos e Ingeniería de Variables
Este notebook utiliza el módulo estructurado `src/data/` para realizar la carga limpia de los datos de IBM Telco Customer Churn, aplicar la división estratificada en Entrenamiento, Validación y Prueba, y configurar el preprocesador reproducible mediante `ColumnTransformer` de Scikit-Learn.

In [ ]:
import os
import sys
import pandas as pd
import joblib

# Asegurar que podemos importar desde src
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.data.data_loader import load_data
from src.data.preprocessing import split_and_preprocess

print("Módulos importados correctamente.")

## 1. Carga Limpia de Datos

In [ ]:
# Ruta al dataset original
raw_filepath = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Llamar a la función modular de carga
df = load_data(raw_filepath)
print(f"Dataset cargado y limpio: {df.shape[0]} registros, {df.shape[1]} columnas")
df.head()

## 2. Partición Estratificada y Preprocesamiento
Para evitar cualquier filtración de datos (*data leakage*), dividimos los datos en entrenamiento (64%), validación (16%) y prueba (20%) y ajustamos el preprocesador (`preprocessor.joblib`) únicamente en el conjunto de entrenamiento. Luego exportamos las particiones resultantes a `data/processed/`.

In [ ]:
# Ejecutar la partición y el ajuste del preprocesador
X_train, X_val, X_test, y_train, y_val, y_test, preprocesador = split_and_preprocess(
    df, 
    output_dir="../data/processed"
)

## 3. Inspeccionar el Preprocesamiento
Verifiquemos cómo se transforman los datos tras pasar por el preprocesador.

In [ ]:
# Transformar una muestra del conjunto de entrenamiento
X_train_trans = preprocesador.transform(X_train)
print(f"Dimensiones de X_train transformado: {X_train_trans.shape}")

# Obtener nombres de las columnas resultantes
columnas_num = preprocesador.transformers_[0][2]
columnas_cat = preprocesador.transformers_[1][2]
encoder = preprocesador.named_transformers_["cat"].named_steps["encoder"]
cat_features = encoder.get_feature_names_out(columnas_cat).tolist()

feature_names = columnas_num + cat_features
print(f"Total de características tras codificación One-Hot: {len(feature_names)}")
print("Primeras 10 características:", feature_names[:10])